In [0]:
# ---------------------------------------------
# GOLD LAYER: LOAD CLEANED SILVER DATA
# ---------------------------------------------
# Purpose:
# This step reads the cleaned and validated dataset from the Silver layer
# into a Spark DataFrame for business-level aggregations and KPI generation.
#
# Why this is needed:
# - Acts as the input for all Gold layer transformations
# - Ensures only high-quality, curated data is used for analytics
# - Maintains separation between transformation (Silver) and analytics (Gold)

df = spark.table("rides_catalog.silver.rides_silver")

In [0]:
# ---------------------------------------------
# GOLD LAYER: DAILY RIDE & REVENUE METRICS
# ---------------------------------------------
# Purpose:
# This transformation generates daily business KPIs from cleaned ride data.
#
# Why this is needed:
# - Provides business insights on ride demand over time
# - Tracks revenue trends using fare aggregation
# - Enables reporting and dashboarding for stakeholders
#
# Output Metrics:
# - total_rides: number of ride events per day
# - total_fare: total revenue generated per day

from pyspark.sql.functions import to_date, sum, count

gold_daily = df.withColumn("date", to_date("timestamp"))\
              .groupBy("date")\
              .agg(count("*").alias("total_rides"),\
                   sum("fare_amount").alias("total_fare"))
                   

In [0]:
%sql

-- ---------------------------------------------
-- GOLD LAYER: SCHEMA CREATION
-- ---------------------------------------------
-- Purpose:
-- This schema represents the Gold layer in the Medallion Architecture.
-- It stores aggregated, business-ready datasets used for analytics and reporting.
--
-- Why this is needed:
-- - Separates business-level metrics from raw and cleaned data
-- - Serves as the final layer for dashboards and reporting tools
-- - Enables structured analytics on curated datasets
--
-- Data Characteristics:
-- - Aggregated data (KPIs, summaries)
-- - Optimized for reporting and BI consumption
-- - No raw or intermediate transformations

create schema if not exists rides_catalog.gold;

In [0]:
# ---------------------------------------------
# GOLD LAYER: WRITE DAILY BUSINESS METRICS
# ---------------------------------------------
# Purpose:
# This step persists the aggregated daily KPI dataset into a Delta table
# within the Gold schema of Unity Catalog.
#
# Why this is needed:
# - Stores business-ready metrics for reporting and dashboards
# - Enables downstream consumption by BI tools (e.g., Power BI, Databricks SQL)
# - Acts as the final layer in the Medallion Architecture
#
# Data Characteristics:
# - Aggregated at daily level
# - Contains key business KPIs such as total rides and total revenue
# - Optimized for analytics consumption (not raw processing)

gold_daily.write.format("delta").mode("overwrite").saveAsTable("rides_catalog.gold.gold_daily")

In [0]:
# ---------------------------------------------
# GOLD LAYER: CITY-LEVEL RIDE DEMAND METRICS
# ---------------------------------------------
# Purpose:
# This transformation generates business insights by calculating
# the total number of rides per pickup city.
#
# Why this is needed:
# - Helps identify high-demand geographic areas
# - Supports operational and business decision-making
# - Enables location-based analytics and reporting
#
# Output Metric:
# - total_rides: number of rides originating from each city


gold_city = df.groupBy("pickup_city").agg(count("*").alias("total_rides"))

In [0]:
# ---------------------------------------------
# GOLD LAYER: WRITE CITY-LEVEL METRICS
# ---------------------------------------------
# Purpose:
# This step persists the city-level ride demand metrics into a Delta table
# within the Gold schema of Unity Catalog.
#
# Why this is needed:
# - Stores aggregated business insights for geographic analysis
# - Enables reporting on ride demand by pickup location
# - Supports dashboards and business decision-making
#
# Data Characteristics:
# - Aggregated by pickup city
# - Contains total ride counts per city
# - Optimized for analytics and reporting workloads

gold_city.write.format("delta").mode("overwrite").saveAsTable("rides_catalog.gold.gold_city")